In [1]:
import os
import ast
import re
from itertools import chain

from dotenv import load_dotenv
from tqdm import tqdm
import pandas as pd
from openai import OpenAI

# Configuration
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
org_key = os.getenv("OPENAI_ORG_KEY")
project_key = os.getenv("OPENAI_PROJECT_KEY")

client = OpenAI(
  api_key=api_key,
  organization=org_key,
  project=project_key,
)

In [2]:
# ----------------------- FUNCTIONS -----------------------
def call_API(input_text):
  response = client.responses.create(
    model="gpt-5.1-2025-11-13",
    tools=[{"type": "web_search"}],
    tool_choice="auto",
    include=["web_search_call.action.sources"],
    temperature=1.0,
    input=input_text,
  )
  return response

In [3]:
# ----------------------- LISTS -----------------------
list_of_link_types = [
    "T1. Rules/Policies/Regulations",
    "T2. Center for Teaching & Learning",
    "T3. AI Institute/Initiative/Center",
    "T4. Library",
    "T5. Academic Integrity",
    "T6. AI Committee",
    "T7. Other Relevant Links",
]

In [4]:
# ----------------------- PROMPTS -----------------------
prompt_for_links = """Find official university or system-level links for [UNIVERSITY] which focus on AI, looking only at this type of link: [LINK_TYPE].

- The links should focus, mention, or pertain to AI.
- The links should be current, up-to-date links.
- Links to relevant PDFs are acceptable.
- Aim to be comprehensive and find all relevant links.
- Return the links in bullet point format. Do not include extra information.
- If the university pages do not include information about AI, return "None"."""

In [5]:
# get list of universities from ICAD-79
df = pd.read_csv("./data/ICAD-79.csv")
list_of_universities = df["Institution"].tolist()
print(list_of_universities)

['Texas A&M University', 'University of Texas at Austin', 'University of North Carolina at Chapel Hill', 'University of South Florida', 'University of Florida', 'Georgia Southern University', 'Southern University and A & M College', 'University of South Alabama', 'Jackson State University', 'University of Wyoming', 'University of California, Berkeley', 'University of Washington-Seattle', 'Arizona State University', 'California State University, Long Beach', 'San José State University', 'Portland State University', 'University of Colorado Colorado Springs', 'University of Michigan at Ann Arbor', 'University of Illinois Urbana-Champaign', 'The Ohio State University', 'Iowa State University', 'Illinois State University', 'Wichita State University', 'Northern Illinois University', 'Ball State University', 'Stony Brook University', 'University of New Hampshire', 'University at Buffalo', 'Binghamton University', 'University of Massachusetts at Dartmouth', 'Rowan University', 'Kean University

In [6]:
records = []
for u in tqdm(list_of_universities, desc="Universities"):
  row_data = {"Institution": u}

  for idx, lt in enumerate(list_of_link_types, start=1):
    # link index
    col_name = "links-"+lt.split(".")[0]

    # Prompt + API Call
    prompt = prompt_for_links.replace("[UNIVERSITY]", u).replace("[LINK_TYPE]", lt.split(". ")[1])
    # print(prompt)
    response = call_API(input_text=prompt)
    output = response.output_text
    # print(output)

    # Add to row
    row_data[col_name] = output

  records.append(row_data)

# Build df
temp_df = pd.DataFrame(records)
df = pd.merge(df, temp_df, on="Institution", how="outer")

Universities: 100%|██████████| 79/79 [51:50<00:00, 39.37s/it] 


In [7]:
# save to csv
df.to_csv("./data/ICAD-79-with-links.csv", index=False)
print("✅ Saved results to ./data/ICAD-79-with-links.csv")

✅ Saved results to ./data/ICAD-79-with-links.csv
